# External Validation on INbreast (ROI-cropped Attention U-Net trained on CBIS-DDSM)

Cross-dataset test of the lesion-delineation model: the ROI-cropped Attention U-Net **trained on
CBIS-DDSM** is applied, **without any retraining**, to INbreast mass lesions. INbreast differs in
scanner (full-field digital), population, and annotation style, so this measures generalization.

- **Images:** INbreast DICOMs (14-bit, normalized to 8-bit).
- **Ground truth:** mass ROI polygons from the OsiriX-plist XML annotations, rasterized to masks.
- **Protocol (identical to CBIS ROI):** per-mass lesion-centered crop from full resolution (bbox x1.8)
  -> 256x256 -> CLAHE -> predict -> Dice / IoU / pointing-game.


In [ ]:
# ---- GPU: pin to RTX ----
import os
os.environ.setdefault('DML_VISIBLE_DEVICES', '0')
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, '| GPUs:', gpus)
if gpus:
    try:
        tf.config.set_visible_devices([gpus[0]], 'GPU'); tf.config.experimental.set_memory_growth(gpus[0], True)
    except Exception as e: print('gpu cfg:', e)
print('Visible:', tf.config.get_visible_devices('GPU'))

In [ ]:
import glob, re, plistlib
import cv2, numpy as np, matplotlib.pyplot as plt, pydicom
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2

In [ ]:
# ============================================================ CONFIG
INB_ROOT = r'D:/INbreast Release 1.0'
XMLDIR = INB_ROOT + '/AllXML'; DCMDIR = INB_ROOT + '/AllDICOMs'
WEIGHTS = 'G:/My Drive/VS_CC/models/cbis_roi_seg.weights.h5'
SAVE_DIR = 'G:/My Drive/VS_CC/models'
SZ = 256; MARGIN = 1.8; MASK_THRESH = 127

## 1. Rebuild the CBIS ROI model + load weights

In [ ]:
def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.Conv2D(f, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    return x
def attn_gate(skip, gating, inter):
    tx = layers.Conv2D(inter, 1)(skip); pg = layers.Conv2D(inter, 1)(gating)
    r = layers.Activation('relu')(layers.Add()([tx, pg])); psi = layers.Conv2D(1, 1, activation='sigmoid')(r)
    return layers.Multiply()([skip, psi])
NAMES = ['block_1_expand_relu', 'block_3_expand_relu', 'block_6_expand_relu', 'block_13_expand_relu', 'block_16_project']
def build_model(sz=SZ):
    inp = layers.Input((sz, sz, 3)); x = layers.Rescaling(1./127.5, offset=-1)(inp)
    base = MobileNetV2(input_tensor=x, include_top=False, weights='imagenet')
    s128, s64, s32, s16, s8 = [base.get_layer(n).output for n in NAMES]
    def up(x, skip, f):
        x = layers.UpSampling2D()(x); a = attn_gate(skip, x, f); return conv_block(layers.Concatenate()([x, a]), f)
    d = up(s8, s16, 256); d = up(d, s32, 128); d = up(d, s64, 64); d = up(d, s128, 32)
    d = conv_block(layers.UpSampling2D()(d), 16)
    return Model(inp, layers.Conv2D(1, 1, activation='sigmoid')(d))
model = build_model(); model.load_weights(WEIGHTS)
print('loaded CBIS ROI weights:', os.path.basename(WEIGHTS))

## 2. INbreast loaders (DICOM + XML mass polygons) + per-mass crop

In [ ]:
_clahe = cv2.createCLAHE(2.0, (8, 8))
dcms = {}
for f in glob.glob(DCMDIR + '/*.dcm'):
    m = re.match(r'(\d+)_', os.path.basename(f))
    if m: dcms[m.group(1)] = f
print('DICOMs indexed:', len(dcms))

def read_dcm(path):
    a = pydicom.dcmread(path).pixel_array.astype(np.float32)
    return ((a - a.min()) / (a.max() - a.min() + 1e-6) * 255).astype(np.uint8)
def parse_masses(xml_path):
    with open(xml_path, 'rb') as fh: pl = plistlib.load(fh, fmt=plistlib.FMT_XML)
    out = []
    for r in pl['Images'][0]['ROIs']:
        if str(r.get('Name', '')).strip().lower() == 'mass':
            pts = [tuple(map(float, re.findall(r'[-\d.]+', s)[:2])) for s in r.get('Point_px', [])]
            pts = [p for p in pts if len(p) == 2]
            if len(pts) >= 3: out.append(np.array(pts, np.int32))
    return out
def crop_around_bbox(img, full_mask, poly, margin=MARGIN, out=SZ):
    r0, r1 = poly[:, 1].min(), poly[:, 1].max(); c0, c1 = poly[:, 0].min(), poly[:, 0].max()
    cy, cx = (r0 + r1) // 2, (c0 + c1) // 2; half = int(max(r1 - r0, c1 - c0) * margin / 2) + 8
    H, W = img.shape
    ri0, ri1 = max(0, cy - half), min(H, cy + half); ci0, ci1 = max(0, cx - half), min(W, cx + half)
    ci = img[ri0:ri1, ci0:ci1]; cm = full_mask[ri0:ri1, ci0:ci1]
    if ci.size == 0: return None, None
    ci = _clahe.apply(cv2.resize(ci, (out, out)))
    cm = (cv2.resize(cm * 255, (out, out), interpolation=cv2.INTER_NEAREST) >= 127).astype(np.uint8)
    return np.stack([ci] * 3, -1).astype(np.float32), cm

# build per-mass crop set
X, Y, meta = [], [], []
for xp in sorted(glob.glob(XMLDIR + '/*.xml')):
    iid = os.path.splitext(os.path.basename(xp))[0]
    if iid not in dcms: continue
    try: masses = parse_masses(xp)
    except Exception: continue
    if not masses: continue
    img = read_dcm(dcms[iid]); H, W = img.shape
    full = np.zeros((H, W), np.uint8)
    for poly in masses: cv2.fillPoly(full, [poly], 1)
    for poly in masses:
        ci, cm = crop_around_bbox(img, full, poly)
        if ci is not None and cm.sum() > 0:
            X.append(ci); Y.append(cm[..., None]); meta.append(iid)
X = np.array(X, np.float32); Y = np.array(Y, np.uint8)
print('INbreast mass crops:', X.shape, '| images:', len(set(meta)), '| coverage %.3f' % Y.mean())

### 2b. DIAGNOSTIC - INbreast crop + mass mask

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(12, 3.2), dpi=110)
for a, i in zip(ax, np.random.RandomState(0).choice(len(X), 4, replace=False)):
    a.imshow(X[i][..., 0], cmap='gray'); a.contour(Y[i, :, :, 0], colors='lime', linewidths=1.3); a.axis('off')
plt.tight_layout(); plt.show()

## 3. External-validation metrics (CBIS-trained model, no retraining)

In [ ]:
P = model.predict(X, batch_size=4, verbose=0)[..., 0]
def per_image(y, p, thr=0.5):
    d, io, hit = [], [], []
    for i in range(len(y)):
        gt = y[i, :, :, 0].astype(bool); pm = p[i] >= thr
        inter = np.logical_and(gt, pm).sum(); s = gt.sum() + pm.sum(); u = np.logical_or(gt, pm).sum()
        d.append(2*inter/s if s else 1.0); io.append(inter/u if u else 1.0)
        yy, xx = np.unravel_index(int(np.argmax(p[i])), p[i].shape); hit.append(int(gt[yy, xx]))
    return np.array(d), np.array(io), np.array(hit)
d, io, hit = per_image(Y, P)
print(f'INbreast EXTERNAL  Dice {d.mean():.3f} +/- {d.std():.3f} (median {np.median(d):.3f}) | '
      f'IoU {io.mean():.3f} | pointing-game {hit.mean()*100:.1f}%  (n={len(d)})')
print(f'   Dice>0.5 rate {np.mean(d > 0.5)*100:.1f}% | Dice>0.7 rate {np.mean(d > 0.7)*100:.1f}%')

### 3b. Overlays: INbreast crop + GT (green) + prediction (red)

In [ ]:
order = np.argsort(-d)[:6]
fig, ax = plt.subplots(1, 6, figsize=(18, 3.2), dpi=110)
for a, idx in zip(ax, order):
    a.imshow(X[idx][..., 0], cmap='gray')
    a.contour(Y[idx, :, :, 0], colors='lime', linewidths=1.2)
    a.contour((P[idx] >= 0.5).astype(float), colors='red', linewidths=1.0)
    a.set_title(f'Dice {d[idx]:.2f}', fontsize=9); a.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'inbreast_extval_overlays.png'), dpi=200, bbox_inches='tight'); plt.show()

## 4. Summary + save

In [ ]:
summary = (
    f"EXTERNAL VALIDATION - the ROI-cropped Attention U-Net trained on CBIS-DDSM, applied without "
    f"retraining to INbreast mass lesions (per-mass crops from full-res DICOM, margin x{MARGIN}, {SZ}x{SZ}): "
    f"Dice {d.mean():.3f} +/- {d.std():.3f} (median {np.median(d):.3f}), IoU {io.mean():.3f}, "
    f"pointing-game {hit.mean()*100:.1f}%, Dice>0.5 {np.mean(d > 0.5)*100:.1f}%, Dice>0.7 {np.mean(d > 0.7)*100:.1f}% "
    f"over {len(d)} INbreast mass lesions ({len(set(meta))} images). "
    f"(In-domain CBIS test was Dice 0.871.)"
)
print(summary)
open(os.path.join(SAVE_DIR, 'inbreast_extval_summary.txt'), 'w').write(summary)
print('\nSaved to', SAVE_DIR)